# Final evidence defense and transfer analysis

Analysis only. No model loading, attack generation or Slurm submission occurs here.
Only strictly merged version-1 final runs are loaded. Public and external data,
source models, attack families, epsilon and transformations remain separate.
Empty output means experiments have not been copied here, not zero attack success.
The candidate has no established efficacy until these experiments are run.

Collected Donut standard attacks are exported as `outputs/donut_attack/donut_collected_external_v1/merged/records.parquet`, not `attacks.parquet`.


In [ ]:
from pathlib import Path
import sys, json, os
import pandas as pd
import matplotlib.pyplot as plt
ROOT = Path.cwd()
if not (ROOT / 'src').is_dir(): ROOT = ROOT.parent
assert (ROOT / 'src/final_protocol').is_dir(), 'Run from repository or notebooks directory'
sys.path.insert(0, str(ROOT / 'src'))
from final_protocol.reporting import summarize, paired_bootstrap, matched_source_intersection
RUN_ROOT = Path(os.environ.get('PP_FINAL_RUN_ROOT', str(ROOT / 'outputs/final_protocol')))
EXPORT = ROOT / 'outputs/final_analysis'
EXPORT.mkdir(parents=True, exist_ok=True)
frames = []
for path in sorted(RUN_ROOT.glob('*/merged/records.parquet')):
    summary = json.loads((path.parent / 'summary.json').read_text())
    if summary['failures'] != 0: raise ValueError(f'Failed evaluations in {path}')
    if not (path.parent / 'final_summary.csv').exists(): raise ValueError(f'Final validation absent: {path}')
    frame = pd.read_parquet(path)
    if len(frame) != summary['records']: raise ValueError('Record count drift')
    frame['run_id'] = path.parent.parent.name
    frames.append(frame)
records = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
print(f'{len(records)} verified evaluations from {len(frames)} runs')


In [ ]:
tables = []
for run_id, group in records.groupby('run_id') if not records.empty else []:
    table = summarize(group)
    table.insert(0, 'run_id', run_id)
    tables.append(table)
summary = pd.concat(tables, ignore_index=True) if tables else pd.DataFrame()
summary.to_csv(EXPORT / 'all_metrics.csv', index=False)
if not summary.empty:
    display(summary.loc[summary.example_type.eq('clean'), ['run_id','dataset_name','model_key','condition','n','clean_accuracy','coverage','selective_accuracy','mean_seconds','mean_vlm_calls','mean_ocr_pages']])
    display(summary.loc[summary.example_type.eq('attack'), ['run_id','dataset_name','model_key','source_model','attack_type','epsilon_255','condition','source_eligible_n','source_eligible_asr','both_clean_correct_n','both_clean_correct_asr','correct_recovery','abstention_rate']])
else:
    print('No final results available. Run the terminal experiments before interpreting tables.')


## Matched comparisons and uncertainty

Intervals resample document differences, not individual repeated evaluations.
Only same-run, exact-matched condition pairs are compared. If two runs overlap,
keep them separate until their protocols and sources have been audited.


In [ ]:
intervals = []
strata = ['run_id','dataset_name','model_key','source_model','attack_type','epsilon_255','example_type']
if not records.empty:
    for key, group in records.groupby(strata, dropna=False):
        for baseline in ['none','jpeg','resize_vote','selective_multiview','semantic_consistency','ocr_only','full_ocr','full_crop']:
            if {'evidence',baseline} <= set(group.condition):
                metric = 'correct' if key[-1] == 'clean' else 'targeted'
                ci = paired_bootstrap(group, 'evidence', baseline, metric=metric)
                intervals.append(dict(zip(strata,key),baseline=baseline,metric=metric,**ci))
pd.DataFrame(intervals).to_csv(EXPORT / 'paired_differences.csv', index=False)
intersection_tables = []
if not records.empty:
    for run_id, group in records.groupby('run_id'):
        matched = matched_source_intersection(group)
        if not matched.empty:
            table = summarize(matched); table.insert(0,'run_id',run_id)
            intersection_tables.append(table)
intersection = pd.concat(intersection_tables,ignore_index=True) if intersection_tables else pd.DataFrame()
intersection.to_csv(EXPORT / 'matched_source_intersection.csv',index=False)


In [ ]:
if not summary.empty:
    attacked = summary.loc[summary.example_type.eq('attack')]
    for key, group in attacked.groupby(['run_id','dataset_name','model_key','source_model','attack_type']):
        fig, ax = plt.subplots(figsize=(10,5))
        for condition, values in group.groupby('condition'):
            values = values.sort_values('epsilon_255')
            ax.plot(values.epsilon_255,values.accepted_targeted_asr,marker='o',label=condition)
        ax.set(xlabel='Epsilon (0–255 pixels)',ylabel='Accepted targeted ASR / all attempts',ylim=(0,1),title=' | '.join(map(str,key)))
        ax.legend(bbox_to_anchor=(1.02,1),loc='upper left'); fig.tight_layout()
        fig.savefig(EXPORT / ('asr_'+'_'.join(map(str,key))+'.png'),dpi=220,bbox_inches='tight'); plt.close(fig)
    clean = summary.loc[summary.example_type.eq('clean')]
    for key, group in clean.groupby(['run_id','dataset_name','model_key']):
        fig, ax = plt.subplots(figsize=(7,5))
        ax.scatter(group.coverage,group.selective_risk)
        for row in group.itertuples(): ax.annotate(row.condition,(row.coverage,row.selective_risk))
        ax.set(xlabel='Clean coverage',ylabel='Selective risk',xlim=(0,1.03),ylim=(0,1),title=' | '.join(key))
        fig.tight_layout(); fig.savefig(EXPORT / ('utility_'+'_'.join(key)+'.png'),dpi=220); plt.close(fig)
print('CSV and PNG exports:', EXPORT)


## Adaptive budget and perceptual audits

A low success rate with flat surrogate losses or discrete query feedback is an
attack limitation. Availability attacks must distinguish already rejected clean
inputs from newly rejected ones. These traces are diagnostics, not certification.


In [ ]:
curves=[]; perceptual=[]
if not records.empty:
    for row in records.to_dict('records'):
        curve=row.get('budget_curve')
        if isinstance(curve,(list,tuple)) or hasattr(curve,'tolist'):
            for point in curve:
                curves.append({k:row[k] for k in ['run_id','dataset_name','record_id','epsilon_255','attack_type']} | dict(point))
        metrics=row.get('per_page_metrics')
        if isinstance(metrics,(list,tuple)) or hasattr(metrics,'tolist'):
            for page,values in enumerate(metrics):
                perceptual.append({k:row[k] for k in ['run_id','dataset_name','record_id','epsilon_255','attack_type']} | dict(page=page,**values))
pd.DataFrame(curves).to_csv(EXPORT/'adaptive_budget_traces.csv',index=False)
pd.DataFrame(perceptual).to_csv(EXPORT/'adaptive_perceptual.csv',index=False)
